In [718]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.append(str(root / "src"))

In [768]:
import torch
from torch import tensor
import torch.nn.functional as F
import math

In [769]:
prompt = [1, 2, 3]
completion = [4, 5, 6]
tokens = prompt + completion
x = torch.tensor(tokens[:-1])
y = torch.tensor(tokens[1:])
print(x, y)

tensor([1, 2, 3, 4, 5]) tensor([2, 3, 4, 5, 6])


In [770]:
vocab = 10
logits = torch.randn(len(x), vocab, requires_grad=True)
loss = F.cross_entropy(logits, y)
loss

tensor(2.3131, grad_fn=<NllLossBackward0>)

In [771]:
logits.grad = None
loss.backward()
logits.grad

tensor([[ 0.0414,  0.0071, -0.1864,  0.0482,  0.0133,  0.0265,  0.0028,  0.0033,
          0.0083,  0.0354],
        [ 0.0326,  0.0078,  0.0561, -0.1921,  0.0044,  0.0371,  0.0103,  0.0103,
          0.0150,  0.0185],
        [ 0.0418,  0.0178,  0.0130,  0.0282, -0.1939,  0.0017,  0.0399,  0.0328,
          0.0137,  0.0048],
        [ 0.0335,  0.0221,  0.0175,  0.0175,  0.0040, -0.1535,  0.0114,  0.0052,
          0.0148,  0.0276],
        [ 0.0021,  0.0510,  0.0029,  0.0046,  0.0071,  0.0006, -0.1009,  0.0206,
          0.0102,  0.0018]])

In [772]:
math.log(vocab)

2.302585092994046

In [724]:
prompt_len = len(prompt)
keep = torch.arange(len(y)) >= prompt_len - 1
keep

tensor([False, False,  True,  True,  True])

In [725]:
loss = F.cross_entropy(logits[keep], y[keep])
loss

tensor(1.9507, grad_fn=<NllLossBackward0>)

In [726]:
logits.grad = None
loss.backward()
logits.grad

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000],
        [ 0.0364,  0.0071,  0.0550,  0.0046, -0.2800,  0.0189,  0.0660,  0.0033,
          0.0816,  0.0072],
        [ 0.0047,  0.0708,  0.0875,  0.0243,  0.0213, -0.3046,  0.0398,  0.0138,
          0.0184,  0.0239],
        [ 0.0217,  0.0382,  0.0338,  0.0465,  0.0119,  0.0444, -0.2639,  0.0252,
          0.0294,  0.0129]])

In [727]:
y_masked = y.masked_fill(~keep, -100)
y_masked

tensor([-100, -100,    4,    5,    6])

In [728]:
loss = F.cross_entropy(logits, y_masked, ignore_index=-100)
loss

tensor(1.9507, grad_fn=<NllLossBackward0>)

In [729]:
logits.grad = None
loss.backward()
logits.grad

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000],
        [ 0.0364,  0.0071,  0.0550,  0.0046, -0.2800,  0.0189,  0.0660,  0.0033,
          0.0816,  0.0072],
        [ 0.0047,  0.0708,  0.0875,  0.0243,  0.0213, -0.3046,  0.0398,  0.0138,
          0.0184,  0.0239],
        [ 0.0217,  0.0382,  0.0338,  0.0465,  0.0119,  0.0444, -0.2639,  0.0252,
          0.0294,  0.0129]])

### reverse string task

In [730]:
from tokenizer import CharTokenizer
import string

tok = CharTokenizer(f">{string.ascii_lowercase}")

In [731]:
tok.ctoi

{'>': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26}

In [732]:
from main import GPT, GPTConfig, small_cfg, device
from dataclasses import replace

In [733]:
w = 'hello'
L = len(w)
s = w + ">" + w[::-1]
s

'hello>olleh'

In [734]:
ids = tok.encode(s)
x = tensor(ids[:-1]); y = tensor(ids[1:])
x, y

(tensor([ 8,  5, 12, 12, 15,  0, 15, 12, 12,  5]),
 tensor([ 5, 12, 12, 15,  0, 15, 12, 12,  5,  8]))

In [735]:
print(tok.decode(x.tolist()))
print(tok.decode(y.tolist()))

hello>olle
ello>olleh


In [736]:
prompt_len = L + 1
keep = torch.arange(len(y)) >= prompt_len - 1
keep

tensor([False, False, False, False, False,  True,  True,  True,  True,  True])

In [737]:
tok.decode(y[keep].tolist())

'olleh'

In [738]:
y_masked = y.masked_fill(~keep, -100)
y_masked

tensor([-100, -100, -100, -100, -100,   15,   12,   12,    5,    8])

In [739]:
import random
"".join(random.choices(string.ascii_lowercase, k=L))

'lblxy'

In [740]:
def get_batch(B, mask = True):
    words = ["".join(random.choices(string.ascii_lowercase, k=L)) for _ in range(B)]
    ids = torch.tensor([tok.encode(w + ">" + w[::-1]) for w in words])  # [B, 11]
    x = ids[:, :-1]                                                     # [B, 10]
    y = ids[:, 1:]                                                      # [B, 10]
    if mask:
        y = y.masked_fill(~keep, -100)
    return x.to(device), y.to(device)


In [741]:
x, y = get_batch(4)
print(x.shape, y.shape)
print(tok.decode(x[0].tolist()))

torch.Size([4, 10]) torch.Size([4, 10])
fguta>atug


In [742]:
cfg = replace(small_cfg, vocab_size=tok.vocab_size, block_size=10)
model = GPT(cfg).to(device)
sum(p.numel() for p in model.parameters())

32224

In [743]:
x, y = get_batch(cfg.batch_size)
logits, _ = model(x)
logits.shape          # [B, 10, 27]

torch.Size([32, 10, 27])

In [744]:
from einops import rearrange
loss = F.cross_entropy(
    rearrange(logits, "b t v -> (b t) v"),
    rearrange(y, "b t -> (b t)"),
    ignore_index=-100,
)
loss

tensor(3.3017, device='cuda:0', grad_fn=<NllLossBackward0>)

In [745]:
math.log(27)

3.295836866004329

In [746]:
def loss_fn(logits, y):
    return F.cross_entropy(
        rearrange(logits, "b t v -> (b t) v"),
        rearrange(y, "b t -> (b t)"),
        ignore_index=-100,
    )


In [747]:
@torch.no_grad()
def evaluate(n=200):
    model.eval()
    words = ["".join(random.choices(string.ascii_lowercase, k=L)) for _ in range(n)]
    prompts = torch.tensor([tok.encode(w + ">") for w in words]).to(device)  # [n, 6]
    out = model.generate(prompts, max_new_tokens=L, temperature=0.0)         # [n, 11]
    preds = [tok.decode(row.tolist()) for row in out[:, -L:]]
    acc = sum(p == w[::-1] for p, w in zip(preds, words)) / n
    model.train()
    return acc, list(zip(words, preds))[:5]


In [748]:
# before training
evaluate()

(0.0,
 [('drnii', '>>>>>'),
  ('banow', '>>>>>'),
  ('zezok', '>>>>>'),
  ('ezfwr', '>>>>>'),
  ('fxbvb', '>>>>>')])

In [749]:
def train(mask = True):
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

    for step in range(2001):
        x, y = get_batch(cfg.batch_size, mask=mask)
        logits, _ = model(x)
        loss = loss_fn(logits, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step % 200 == 0:
            print(step, loss.item())


In [750]:
train()

0 3.3012077808380127
200 0.3677480220794678
400 0.048322729766368866
600 0.019347146153450012
800 0.010430539026856422
1000 0.0067010014317929745
1200 0.004674849566072226
1400 0.0034642256796360016
1600 0.002623924519866705
1800 0.0020713137928396463
2000 0.0017044034320861101


In [751]:
# after training
evaluate()

(1.0,
 [('xgufm', 'mfugx'),
  ('ojzqi', 'iqzjo'),
  ('kypfe', 'efpyk'),
  ('gzrxd', 'dxrzg'),
  ('brtux', 'xutrb')])

In [752]:
model = GPT(cfg).to(device)
train(mask=False)

0 3.31911039352417
200 1.7650314569473267
400 1.3537225723266602
600 1.3203303813934326
800 1.3217111825942993
1000 1.3157570362091064
1200 1.307798147201538
1400 1.3125015497207642
1600 1.3072385787963867
1800 1.3076376914978027
2000 1.3111021518707275


In [753]:
evaluate()

(1.0,
 [('xpazb', 'bzapx'),
  ('kjxdx', 'xdxjk'),
  ('qplat', 'talpq'),
  ('scpdf', 'fdpcs'),
  ('qyywu', 'uwyyq')])

In [754]:
math.log(27)

3.295836866004329

In [755]:
4/10 * math.log(26)

1.3032386152085929

In [756]:
x, y = get_batch(4096)          # masked
logits, _ = model(x)
loss_fn(logits, y)

tensor(0.0045, device='cuda:0', grad_fn=<NllLossBackward0>)

### variable length

In [757]:
tok = CharTokenizer(f".>{string.ascii_lowercase}")   # '.' = 0, '>' = 1, letters 2..27
tok.vocab_size                                        # 28

28

In [758]:
Lmin, Lmax = 3, 8
S = 2 * Lmax + 1    # 17, padded string length
T = S - 1           # 16, x/y length = block_size

In [759]:
w = "abc"
s = (w + ">" + w[::-1]).ljust(S, ".")
ids = torch.tensor(tok.encode(s))
x, y = ids[:-1], ids[1:]
print(tok.decode(x.tolist()))
print(tok.decode(y.tolist()))

abc>cba.........
bc>cba..........


In [760]:
l = len(w)
prompt_len = l + 1                 # word + '>'
start = prompt_len - 1             # first kept position
end = start + l                    # one past the last kept position
pos = torch.arange(T)
keep = (pos >= start) & (pos < end)
tok.decode(y[keep].tolist())       # should be exactly 'cba'

'cba'

In [761]:
def get_batch(B, mask=True):
    Ls = [random.randint(Lmin, Lmax) for _ in range(B)]
    words = ["".join(random.choices(string.ascii_lowercase, k=l)) for l in Ls]
    ids = torch.tensor([tok.encode((w + ">" + w[::-1]).ljust(S, ".")) for w in words])
    x, y = ids[:, :-1], ids[:, 1:]
    if mask:
        Lt = torch.tensor(Ls)
        start = Lt[:, None]                    # prompt_len - 1, as [B, 1]
        end = start + Lt[:, None]              # [B, 1]
        pos = torch.arange(T)                  # [T]
        keep = (pos >= start) & (pos < end)    # [B, T]
        y = y.masked_fill(~keep, -100)
    return x.to(device), y.to(device)


In [762]:
x, y = get_batch(4)
for i in range(x.size(0)):
    print(tok.decode(x[i].tolist()))

pwbqc>cqbwp.....
gmufr>rfumg.....
oze>ezo.........
ltlovw>wvoltl...


In [763]:
cfg = replace(small_cfg, vocab_size=tok.vocab_size, block_size=20)
model = GPT(cfg).to(device)

In [764]:
train()

0 3.3491859436035156
200 1.8556885719299316
400 0.2975824475288391
600 0.08529002964496613
800 0.06863103806972504
1000 0.08422835171222687
1200 0.019098429009318352
1400 0.014277600683271885
1600 0.0109064020216465
1800 0.008704260922968388
2000 0.007136328611522913


In [765]:
@torch.no_grad()
def evaluate(n=200, lmin=None, lmax=None):
    lmin = Lmin if lmin is None else lmin
    lmax = Lmax if lmax is None else lmax
    model.eval()
    correct, samples = 0, []
    for _ in range(n):
        l = random.randint(lmin, lmax)
        w = "".join(random.choices(string.ascii_lowercase, k=l))
        prompt = torch.tensor([tok.encode(w + ">")]).to(device)
        out = model.generate(prompt, max_new_tokens=l, temperature=0.0)
        pred = tok.decode(out[0, -l:].tolist())
        correct += pred == w[::-1]
        samples.append((w, pred))
    model.train()
    return correct / n, samples[:5]


In [766]:
evaluate()

(1.0,
 [('jjrrvqah', 'haqvrrjj'),
  ('hvlk', 'klvh'),
  ('mgckkex', 'xekkcgm'),
  ('eyxp', 'pxye'),
  ('nupajilx', 'xlijapun')])

In [767]:
evaluate(lmin=9, lmax=10)

(0.0,
 [('zgrwqckaz', 'aqwrgzzzz'),
  ('xoswlcdfj', 'flwsoxxxx'),
  ('idgiuawsev', 'wuigdiiiii'),
  ('dkktfpvzdf', 'zftkkddddd'),
  ('hyhlqqjrk', 'rqlhyhhhh')])

Worth unpacking properly, and doing so turns up something I got wrong in the roadmap entry.

**What the model actually has to compute.** With word length `l`, the string is `w[0..l-1]` + `>` at index `l` + the reversal at `l+1..2l`. Position `t` predicts `s[t+1]`, and for `t` in the completion that character is `w[2l-1-t]` — living at input position `2l-1-t`.

So the source position `p` satisfies `p + t = 2l - 1`. Reversal is a **reflection**, not a translation. The offset the head would need is `p - t = 2l - 2t - 1`, which changes with _both_ `t` and `l`. There is no single relative offset that solves it.

**Why that's still easy here.** Since it can't be one rule, it has to be a table — and the table is tiny. Valid `(l, t)` pairs are `t ∈ [l, 2l-1]`, so `l` cases per length:

    l=3: 3   l=5: 5   l=7: 7
    l=4: 4   l=6: 6   l=8: 8      total = 33
    

Thirty-three. Your model has 32k parameters. `>` is a unique token, so one head can locate it and write `l` into the residual stream; from there an MLP only needs to memorize 33 entries mapping `(pos_t, pos_l)` to "attend to position `2l-1-t`". That's not an algorithm, it's a lookup — and it fits comfortably. Variable length raised the case count from 5 to 33 and I assumed that crossed some threshold. It didn't come close.

**Why 9–10 fail completely.** They introduce 19 new `(l, t)` pairs, none in the table. And nothing interpolates: `position_embedding_table` rows 9–19 saw only pad during training, so they sit near their `N(0, 0.02)` init with no learned relationship to rows 3–8. The lookup has no entry and the representation has no structure to fall back on. Hence 0.0, with those correct-chunk-wrong-offset outputs — it's snapping unseen inputs onto the nearest entries it does have.

**The correction.** I wrote in the roadmap that reversal "is exactly what RoPE encodes" and that a flat number means a broken implementation. That's too strong, and I'd rather you not go debugging correct code. RoPE makes attention scores depend on `t - p` and removes the untrained-absolute-rows problem, but the required offset `2l - 2t - 1` still varies per position — RoPE doesn't hand the model a reflection. Length generalization on reverse-and-copy tasks is a known-hard case that positional encoding alone often doesn't fix.

Let me soften that roadmap paragraph — the test is still worth running, it just isn't a pass/fail on your RoPE code.